In [5]:
import torch
from pytorch3d.io import load_objs_as_meshes
from pytorch3d.ops import sample_points_from_meshes

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
object = "models/Sphere_sofa.obj"

In [3]:
# mesh loading
mesh = load_objs_as_meshes([object], device=device)

print("Vertices:", mesh.verts_packed().shape)
print("Faces:", mesh.faces_packed().shape)

/home/mad03633/Assign_1/graphics/lib/python3.10/site-packages/pytorch3d/io/obj_io.py:548: UserWarning: Mtl file does not exist: models/15379_Abstract_Bench_v1.mtl
  warnings.warn(f"Mtl file does not exist: {f}")


Vertices: torch.Size([12938, 3])
Faces: torch.Size([25872, 3])


In [4]:
from pytorch3d.vis.plotly_vis import plot_scene 


fig = plot_scene(
    { "Meshes": { 
        "Original OBJ": mesh 
        } 
    }) 

fig.show()

In [32]:
# points cloud generation
num_samples = 5000
points = sample_points_from_meshes(mesh, num_samples=num_samples)
print("Point cloud:", points.shape)

Point cloud: torch.Size([1, 5000, 3])


In [ ]:
from pytorch3d.structures import Pointclouds
from pytorch3d.vis.plotly_vis import plot_scene

pc_original = Pointclouds(points=[points[0]])

fig = plot_scene({
    "Scene": {
        "Original Point Cloud": pc_original
    }
})

fig.show()

In [35]:
# Gaussian noise
noise_std = 0.01
noise = torch.randn_like(points) * noise_std
noisy_points = points + noise
print("Noisy cloud:", noisy_points.shape)

Noisy cloud: torch.Size([1, 5000, 3])


In [36]:
pc_noisy = Pointclouds(points=[noisy_points[0]])

fig = plot_scene({
    "Scene": {
        "Noisy Point Cloud": pc_noisy
    }
})

fig.show()

In [37]:
from pytorch3d.io import save_ply


save_ply("Sphere_sofa_pointcloud.ply", noisy_points[0])

Task

In [38]:
from pytorch3d.io import load_ply


verts, faces = load_ply("Sphere_sofa_pointcloud.ply")
verts = verts.to(device)

verts = verts[None, :, :]
print(verts.shape)

torch.Size([1, 5000, 3])


In [39]:
# Normalization
center = verts.mean(1)
verts = verts - center
scale = verts.abs().max()
verts = verts / scale
print(verts.shape)

torch.Size([1, 5000, 3])


In [40]:
pc_normalized = Pointclouds(points=[verts[0]])

fig = plot_scene({
    "Scene": {
        "Normalized Point Cloud": pc_normalized
    }
})

fig.show()

In [41]:
# Spherical mesh
from pytorch3d.utils import ico_sphere


src_mesh = ico_sphere(5, device)
print(src_mesh)

In [42]:
# deform
src_verts = src_mesh.verts_packed()

deform_verts = torch.zeros_like(
    src_verts,
    requires_grad=True,
    device=device
)

In [43]:
# optimizer
optimizer = torch.optim.SGD(
    [deform_verts],
    lr=0.1,
    momentum=0.9
)

In [44]:
# Loss func
from pytorch3d.loss import chamfer_distance, mesh_edge_loss, mesh_laplacian_smoothing

w_chamfer = 1.0
w_edge = 1.0
w_laplacian = 0.1

Individual Part

In [48]:
from pytorch3d.structures import Meshes

def runner(w_chamfer, w_edge, w_laplacian, exp_name):

    deform_verts = torch.zeros_like(
        src_mesh.verts_packed(),
        requires_grad=True,
        device=device
    )

    optimizer = torch.optim.SGD(
        [deform_verts],
        lr=0.1,
        momentum=0.9
    )

    for i in range(2000):

        optimizer.zero_grad()

        new_src_mesh = src_mesh.offset_verts(deform_verts)

        sample_src = sample_points_from_meshes(new_src_mesh, 5000)

        loss_chamfer, _ = chamfer_distance(sample_src, verts)
        loss_edge = mesh_edge_loss(new_src_mesh)
        loss_laplacian = mesh_laplacian_smoothing(new_src_mesh)

        loss = (
            w_chamfer * loss_chamfer +
            w_edge * loss_edge +
            w_laplacian * loss_laplacian
        )

        loss.backward()
        optimizer.step()

        if i % 200 == 0:
            print(
                exp_name,
                "iter:", i,
                "loss:", loss.item()
            )

    final_mesh = src_mesh.offset_verts(deform_verts)

    return final_mesh

In [49]:
# Experiment A - Without Regularization
mesh_A = runner(
    w_chamfer=1.0,
    w_edge=0.0,
    w_laplacian=0.0,
    exp_name="Experiment A"
)

Experiment A iter: 0 loss: 0.14996540546417236
Experiment A iter: 200 loss: 0.09151749312877655
Experiment A iter: 400 loss: 0.06539711356163025
Experiment A iter: 600 loss: 0.05431819334626198
Experiment A iter: 800 loss: 0.04676003009080887
Experiment A iter: 1000 loss: 0.04031648486852646
Experiment A iter: 1200 loss: 0.03845331445336342
Experiment A iter: 1400 loss: 0.03341575711965561
Experiment A iter: 1600 loss: 0.03268008679151535
Experiment A iter: 1800 loss: 0.03127923607826233


In [50]:
# Experiment B - Strong Smoothness
mesh_B = runner(
    w_chamfer=1.0,
    w_edge=0.0,
    w_laplacian=10.0,
    exp_name="Experiment B"
)

Experiment B iter: 0 loss: 0.16082283854484558
Experiment B iter: 200 loss: 0.12913911044597626
Experiment B iter: 400 loss: 0.08503955602645874
Experiment B iter: 600 loss: 0.06911934912204742
Experiment B iter: 800 loss: 0.06375371664762497
Experiment B iter: 1000 loss: 0.06050461530685425
Experiment B iter: 1200 loss: 0.0559668093919754
Experiment B iter: 1400 loss: 0.05253506824374199
Experiment B iter: 1600 loss: 0.05172421783208847
Experiment B iter: 1800 loss: 0.04895364120602608


In [51]:
# Experiment C - Balanced
mesh_C = runner(
    w_chamfer=1.0,
    w_edge=0.5,
    w_laplacian=0.05,
    exp_name="Experiment C"
)

Experiment C iter: 0 loss: 0.1511175036430359
Experiment C iter: 200 loss: 0.0908539816737175
Experiment C iter: 400 loss: 0.06530167162418365
Experiment C iter: 600 loss: 0.05346212536096573
Experiment C iter: 800 loss: 0.04773996025323868
Experiment C iter: 1000 loss: 0.04213956370949745
Experiment C iter: 1200 loss: 0.04083741828799248
Experiment C iter: 1400 loss: 0.03810947388410568
Experiment C iter: 1600 loss: 0.034430984407663345
Experiment C iter: 1800 loss: 0.03214721381664276


In [ ]:
# Saving
from pytorch3d.io import save_obj

save_obj("mesh_A_sofa.obj", mesh_A.verts_packed(), mesh_A.faces_packed())
save_obj("mesh_B_sofa.obj", mesh_B.verts_packed(), mesh_B.faces_packed())
save_obj("mesh_C_sofa.obj", mesh_C.verts_packed(), mesh_C.faces_packed())

In [7]:
from pytorch3d.io import load_obj
from pytorch3d.structures import Meshes

def load_mesh(path):

    verts, faces, _ = load_obj(path)

    mesh = Meshes(
        verts=[verts.to(device)],
        faces=[faces.verts_idx.to(device)]
    )

    return mesh


mesh_A = load_mesh("meshes/mesh_A_cup.obj")
mesh_B = load_mesh("meshes/mesh_A_cup.obj")
mesh_C = load_mesh("meshes/mesh_A_cup.obj")

In [8]:
# Visualization
import torch
from pytorch3d.vis.plotly_vis import plot_scene

def shift_mesh(mesh, shift):
    verts = mesh.verts_packed()
    verts = verts + torch.tensor(shift, device=verts.device)
    return mesh.update_padded(verts[None])

mesh_A_shift = shift_mesh(mesh_A, [-3,0,0])
mesh_B_shift = shift_mesh(mesh_B, [0,0,0])
mesh_C_shift = shift_mesh(mesh_C, [3,0,0])

fig = plot_scene({
    "Meshes": {
        "Experiment A": mesh_A_shift,
        "Experiment B": mesh_B_shift,
        "Experiment C": mesh_C_shift
    }
})

fig.show()